In [19]:
import pandas as pd
import numpy as np

import string
import nltk
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.naive_bayes import BernoulliNB, MultinomialNB

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [20]:
df=pd.read_csv("smsspamcollection.csv", sep=',', names=['label', 'message'], encoding='utf-8-sig', engine='python')

In [21]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to C:\Users\Suraj
[nltk_data]     Shah\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [22]:
stop_words=set(stopwords.words('english'))

def preprocess(text):
    text=text.lower()

    text=''.join([char for char in text if char not in string.punctuation])

    words=text.split()

    words=[word for word in words if word not in stop_words]

    return ' '.join(words)

In [ ]:
df['clean_message']=df['message'].apply(preprocess)
df.head()

,label,message,clean_message
0,ham,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis n great ...
1,ham,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry 2 wkly comp win fa cup final tkts 2...
3,ham,U dun say so early hor... U c already then say...,u dun say early hor u c already say
4,ham,"Nah I don't think he goes to usf, he lives aro...",nah dont think goes usf lives around though


In [24]:
df['label']=df['label'].map({'ham': 0, 'spam': 1})

In [25]:
X=df['clean_message']
y=df['label']

X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=0.2, random_state=42)

In [29]:
count_vectorizer=CountVectorizer(binary=True)

X_train_count=count_vectorizer.fit_transform(X_train)
X_test_count=count_vectorizer.transform(X_test)

In [30]:
tfidf_vectorizer=TfidfVectorizer()

X_train_tfidf=tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf=tfidf_vectorizer.transform(X_test)

In [34]:
bnb=BernoulliNB()
bnb.fit(X_train_count, y_train)

y_pred_bnb=bnb.predict(X_test_count)

In [35]:
mnb=MultinomialNB()
mnb.fit(X_train_tfidf, y_train)

y_pred_mnb=mnb.predict(X_test_tfidf)

In [36]:
def evaluate(y_test, y_pred, model_name):
    print(f"\n {model_name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [37]:
evaluate(y_test, y_pred_bnb, "Bernoulli Naive Bayes")
evaluate(y_test, y_pred_mnb, "Multinomial Naive Bayes")


 Bernoulli Naive Bayes
Accuracy: 0.9748878923766816
Precision: 0.991869918699187
Recall: 0.8187919463087249
F1 Score: 0.8970588235294118

Classification Report:
               precision    recall  f1-score   support

           0       0.97      1.00      0.99       966
           1       0.99      0.82      0.90       149

    accuracy                           0.97      1115
   macro avg       0.98      0.91      0.94      1115
weighted avg       0.98      0.97      0.97      1115


 Multinomial Naive Bayes
Accuracy: 0.9713004484304932
Precision: 1.0
Recall: 0.785234899328859
F1 Score: 0.8796992481203008

Classification Report:
               precision    recall  f1-score   support

           0       0.97      1.00      0.98       966
           1       1.00      0.79      0.88       149

    accuracy                           0.97      1115
   macro avg       0.98      0.89      0.93      1115
weighted avg       0.97      0.97      0.97      1115

